In [1]:
pip install -U kaleido

Note: you may need to restart the kernel to use updated packages.


## Required Imports

In [2]:

# Route transformers/tf.keras to Keras 2 (tf_keras). Must be set BEFORE importing tensorflow/transformers.
import os
#os.environ["TF_USE_LEGACY_KERAS"] = "1"

# 这台机器的 HF_HOME 默认指向网络盘 /workspace，小文件+文件锁在网络盘上很慢甚至会卡死，改到本地盘。
os.environ["HF_HOME"] = "/root/.cache/huggingface"

import numpy as np
import tensorflow as tf

# TF 默认一启动就把几乎整块 GPU 显存（这台机器是 24GB）预先占为己用当内部内存池，
# 不是真的都用到了。换过 cuda_malloc_async 分配器还是一样几乎占满、照样 OOM，
# 说明问题是"预先占满导致后面分配不到"，不是分配器实现的问题。
# 改成按需增长显存占用，需要在建任何模型 / 跑任何 GPU 算子之前设置。
_gpus = tf.config.experimental.list_physical_devices('GPU')
for _gpu in _gpus:
    tf.config.experimental.set_memory_growth(_gpu, True)

import gc, sys, glob
from tensorflow.keras import Input
from tensorflow.keras import models as M
from tensorflow.keras import layers as L
from tensorflow.keras import backend as keras
from tensorflow.keras.utils import plot_model
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import fpsample
from transformers import TFBertModel, BertTokenizer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse, parse_qs
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
import socket
import json
import subprocess
import plotly.graph_objects as go

2026-07-27 04:43:52.934771: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-27 04:43:52.960946: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-27 04:43:52.960965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-27 04:43:52.962241: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-27 04:43:52.967163: I tensorflow/core/platform/cpu_feature_guar

# Utility Functions for the model 

In [3]:
def pairwise_distance(xyz1, xyz2):
    n = xyz1.shape[1]
    c = xyz1.shape[2]
    m = xyz2.shape[1]
    # 这里会 materialize 一个 (N, M, 3) 的两两距离张量（比如 4096x4096x3），
    # 换过 cuda_malloc_async 分配器还是会 OOM，说明确实是显存不够而不是碎片化，
    # 挪到 CPU 算（这几千个点的规模对 CPU 来说很快），不影响模型输出结果。
    with tf.device('/CPU:0'):
        xyz1 = tf.tile(tf.reshape(xyz1, (-1,1,n,c)), [1,m,1,1])
        xyz2 = tf.tile(tf.reshape(xyz2, (-1,m,1,c)), [1,1,n,1])
        dist = tf.reduce_sum((xyz1-xyz2)**2, -1)
    return dist

def knn_point(k, xyz1, xyz2):
    dist = -pairwise_distance(xyz1, xyz2)
    val, idx = tf.math.top_k(dist, k)
    return -val, idx

class UniformSampler(tf.keras.layers.Layer):
    def __init__(self, num_points, seed=42, **kwargs):
        super(UniformSampler, self).__init__(**kwargs)
        self.num_points = num_points
        self.seed = seed

    def build(self, input_shape):
        pass

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        data_size = tf.shape(inputs)[1]
        indices = tf.random.uniform(
            shape=(batch_size, self.num_points),
            minval=0,
            maxval=data_size,
            dtype=tf.int32,
            seed=self.seed
        )
        return indices

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.num_points, input_shape[2])

    def get_config(self):
        config = super(UniformSampler, self).get_config()
        config.update({
            "num_points": self.num_points,
            "seed": self.seed
        })
        return config

def sample_and_group(args, nsample):
    xyz, pts, fps_idx = args
    new_xyz = tf.gather_nd(xyz, tf.expand_dims(fps_idx,-1), batch_dims=1)
    new_pts = tf.gather_nd(pts, tf.expand_dims(fps_idx,-1), batch_dims=1)
    _, idx = knn_point(nsample, xyz, new_xyz)
    grouped_pts = tf.gather_nd(pts, tf.expand_dims(idx,-1), batch_dims=1)
    grouped_pts -= tf.tile(tf.expand_dims(new_pts, 2),
                           (1,1,nsample,1))
    new_pts = tf.concat([grouped_pts,
                         tf.tile(tf.expand_dims(new_pts, 2),
                                 (1,1,nsample,1))],
                        axis=-1)
    return new_xyz, new_pts

def LBR(tensor, C, seq_name, use_bias=True, activation=None, LeakyAlpha=0.0):
    x_in = Input(shape=tensor.shape[1:], name=seq_name+'_input')
    x = L.Dense(C, use_bias=use_bias, activation=activation, name=seq_name+'_lin')(x_in)
    if LeakyAlpha==0.0:
        x_out = L.ReLU(name=seq_name+'_ReLU')(x)
    else:
        x_out = L.LeakyReLU(alpha=LeakyAlpha, name=seq_name+'_ReLU')(x)
    model = M.Model(inputs=x_in, outputs=x_out, name=seq_name)
    return model(tensor)

def Self_Attention(tensor, seq_name):
    x_in = Input(shape=tensor.shape[1:], name=seq_name+'_input')
    C = x_in.shape[2]
    W_q = L.Dense(C//4, use_bias=False, activation=None, name=seq_name+'_Q')
    W_k = L.Dense(C//4, use_bias=False, activation=None, name=seq_name+'_K')
    W_v = L.Dense(C, use_bias=False, activation=None, name=seq_name+'_V')
    x_q = W_q(x_in)
    x_k = W_k(x_in)
    W_k.set_weights(W_q.get_weights())
    x_k = L.Lambda(lambda t: tf.transpose(t, perm=(0,2,1)), name=seq_name+'_KT')(x_k)
    x_v = W_v(x_in)
    energy = L.Lambda(lambda ts: tf.matmul(ts[0],ts[1]), name=seq_name+'_matmul1')([x_q, x_k])
    attention = L.Softmax(axis=1, name=seq_name+'_softmax')(energy)
    attention = L.Lambda(lambda t: t / (1e-9 + tf.reduce_sum(t, axis=2, keepdims=True)), name=seq_name+'_l1norm')(attention)
    x_r = L.Lambda(lambda ts: tf.matmul(ts[0],ts[1]), name=seq_name+'_matmul2')([attention, x_v])
    x_r = L.Lambda(lambda ts: tf.subtract(ts[0],ts[1]), name=seq_name+'_subtract')([x_in, x_r])
    x_r = LBR(x_r, C, seq_name+'_LBR', use_bias=True)
    x_out = L.Lambda(lambda ts: tf.add(ts[0],ts[1]), name=seq_name+'_add')([x_in, x_r])
    model = M.Model(inputs=x_in, outputs=x_out, name=seq_name)
    return model(tensor)


def Cross_Attention(args, seq_name):
    E_tensor, D_tensor = args
    xE_in = Input(shape=E_tensor.shape[1:], name=seq_name+'_input-E')
    C = xE_in.shape[2]
    xD_in = Input(shape=D_tensor.shape[1:], name=seq_name+'_input-D')
    out_dim = xD_in.shape[2]
    W_q = L.Dense(C//4, use_bias=False, activation=None, name=seq_name+'_Q')
    W_k = L.Dense(C//4, use_bias=False, activation=None, name=seq_name+'_K')
    W_v = L.Dense(out_dim, use_bias=False, activation=None, name=seq_name+'_V')
    x_q = W_q(xD_in)
    x_k = W_k(xE_in)
    x_k = L.Lambda(lambda t: tf.transpose(t, perm=(0,2,1)), name=seq_name+'_KT')(x_k)
    x_v = W_v(xE_in)
    energy = L.Lambda(lambda ts: tf.matmul(ts[0],ts[1]), name=seq_name+'_matmul1')([x_q, x_k])
    attention = L.Softmax(axis=1, name=seq_name+'_softmax')(energy)
    attention = L.Lambda(lambda t: t / (1e-9 + tf.reduce_sum(t, axis=2, keepdims=True)), name=seq_name+'_l1norm')(attention)
    x_r = L.Lambda(lambda ts: tf.matmul(ts[0],ts[1]), name=seq_name+'_matmul2')([attention, x_v])
    x_r = L.Lambda(lambda ts: tf.subtract(ts[0],ts[1]), name=seq_name+'_subtract')([xD_in, x_r])
    x_r = LBR(x_r, out_dim, seq_name+'_LBR', use_bias=True)
    x_out = L.Lambda(lambda ts: tf.add(ts[0],ts[1]), name=seq_name+'_add')([xD_in, x_r])
    model = M.Model(inputs=[xE_in,xD_in], outputs=x_out, name=seq_name)
    return model([E_tensor,D_tensor])

def copy_and_mapping(tensor, nmul, seq_name):
    x_in = Input(shape=tensor.shape[1:], name=seq_name+'_input')
    x = L.Lambda(lambda t: tf.expand_dims(t, 2), name=seq_name+'_expand')(x_in)
    C = x.shape[-1]//nmul
    x1 = L.Conv2DTranspose(C,(1,nmul),(1,nmul), use_bias=True, activation=None, name=seq_name+'_convT')(x)
    x2 = L.Dense(C, use_bias=True, activation=None, name=seq_name+'_lin')(x)
    x2 = L.Lambda(lambda t: tf.tile(t, [1,1,nmul,1]), name=seq_name+'_tile')(x2)
    x = L.Lambda(lambda ts: tf.add(ts[0],ts[1]), name=seq_name+'_add')([x1, x2])
    npoint = x.shape[1]*x.shape[2]
    x_out = L.Lambda(lambda t: tf.reshape(t, [-1,npoint,t.shape[3]]), name=seq_name+'_reshape')(x)
    model = M.Model(inputs=x_in, outputs=x_out, name=seq_name)
    return model(tensor)

## Point Encoder

In [4]:
def PCT_encoder(xyz):
    x = LBR(xyz, 64, 'E-IN_LBR1', use_bias=False)
    x = LBR(x, 128, 'E-IN_LBR2', use_bias=False)
    fps_idx = UniformSampler(4096)(xyz)
    new_xyz, new_feature = L.Lambda(sample_and_group, arguments={'nsample':32}, name='E-SG1')([xyz, x, fps_idx])
    x = LBR(new_feature, 512, 'E-SG1_LBR1', use_bias=False)
    x = L.Lambda(lambda t: tf.reduce_max(t, axis=2), name='E-SG1_MaxPool')(x)
    fps_idx = UniformSampler(2048)(new_xyz)
    new_xyz, new_feature = L.Lambda(sample_and_group, arguments={'nsample':32}, name='E-SG2')([new_xyz, x, fps_idx])
    x = LBR(new_feature, 1024, 'E-SG2_LBR1', use_bias=False)
    x = L.Lambda(lambda t: tf.reduce_max(t, axis=2), name='E-SG2_MaxPool')(x)
    x1 = Self_Attention(x, 'E-SA1')
    x2 = Self_Attention(x1, 'E-SA2')
    x3 = Self_Attention(x2, 'E-SA3')
    x4 = Self_Attention(x3, 'E-SA4')
    x0 = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='E-SA_Concat')([x1,x2,x3,x4])
    x = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='E-OUT_Concat')([x0,x])
    x = LBR(x, 2048, 'E-OUT_LBR', use_bias=False, LeakyAlpha=0.2)
    x1 = Self_Attention(x, 'E-SA5')
    x2 = Self_Attention(x1, 'E-SA6')
    x3 = Self_Attention(x2, 'E-SA7')
    x4 = Self_Attention(x3, 'E-SA8')
    x0 = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='E-SA_Concat2')([x1,x2,x3,x4])
    x = LBR(x0, 4096, 'E-OUT_LBR1', use_bias=False, LeakyAlpha=0.2)
    output_feats = L.Lambda(lambda t: tf.reduce_max(t, axis=1, keepdims=True), name='E-OUT_MaxPool')(x)
    return output_feats

## Point Decoder

In [5]:
def pct_decoder(input_feats, input_eye_seed):
    m_feats = L.Lambda(lambda x: tf.tile(x, [1,1024,1]), name = 'D-IN_replicate')(input_feats)
    input_eye = input_eye_seed + tf.eye(1024,1024)
    x = L.Dense(4096//4, use_bias=False, activation=None, name='D1-IN')(input_eye)
    x1 = Cross_Attention([m_feats,x] , 'D-STA1')
    x2 = Cross_Attention([m_feats,x1], 'D-STA2')
    x3 = Cross_Attention([m_feats,x2], 'D-STA3')
    x4 = Cross_Attention([m_feats,x3], 'D-STA4')
    x0 = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='D1-STA_Concat')([x1,x2,x3,x4])
    x = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='D1-OUT_Concat')([x0,x])
    m_feats2 = copy_and_mapping(x, 3, 'D1-OUT_CopyAndMapping')
    input_eye2 = input_eye_seed + tf.eye(3072,3072)
    x = L.Dense(1024//4, use_bias=False, activation=None, name='D2-IN')(input_eye2)
    x1 = Cross_Attention([m_feats2,x] , 'D2-STA1')
    x2 = Cross_Attention([m_feats2,x1], 'D2-STA2')
    x3 = Cross_Attention([m_feats2,x2], 'D2-STA3')
    x4 = Cross_Attention([m_feats2,x3], 'D2-STA4')
    x0 = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='D2-STA_Concat')([x1,x2,x3,x4])
    x = L.Lambda(lambda ts: tf.concat(ts, axis=2), name='D2-OUT_Concat')([x0,x])
    x = copy_and_mapping(x, 2, 'D2-OUT_CopyAndMapping')
    x = LBR(x,128, 'D-OUT_LBR1', use_bias=False)
    x = LBR(x,128, 'D-OUT_LBR2', use_bias=False)
    x = LBR(x, 64, 'D-OUT_LBR3', use_bias=False, LeakyAlpha=0.2)
    output_points = L.Dense(3, activation=None, name='D-OUT_lin')(x)
    return output_points

## Bert Text Encoder

In [6]:
def bert_model(input_ids, attention_mask, model_name='bert-base-uncased', max_length=128):
    bert_model = TFBertModel.from_pretrained(model_name, use_safetensors=False)
    bert_model.trainable = False
    bert_outputs = bert_model([input_ids, attention_mask])
    cls_output = bert_outputs.pooler_output
    dense_output = tf.keras.layers.Dense(4096, activation='relu')(cls_output)
    output = tf.expand_dims(dense_output, axis = 1)
    return output

## Building the multi-modal point cloud autoencoder

In [7]:
class PCT_AE_Multimodal:
    def __init__(self, num_input_points=4096, max_length=128, bert_model=bert_model, PCT_encoder=PCT_encoder, pct_decoder=pct_decoder):
        self.num_input_points = num_input_points
        self.max_length = max_length
        self.bert_model = bert_model
        self.PCT_encoder = PCT_encoder
        self.pct_decoder = pct_decoder
        self.model = self.build_model()

    def build_model(self):
        eye_seed = Input(shape=(1, 1), name='input_eye_seed')
        xyz = Input(shape=(self.num_input_points, 3), name='input_points')
        input_ids = Input(shape=(self.max_length,), dtype=tf.int32, name='input_ids')
        attention_mask = Input(shape=(self.max_length,), dtype=tf.int32, name='attention_mask')
        if not self.bert_model or not self.PCT_encoder or not self.pct_decoder:
            raise ValueError("Bert model, PCT encoder, and PCT decoder must be provided.")
        text_encoded = self.bert_model(input_ids, attention_mask)
        cloud_encoded = self.PCT_encoder(xyz)
        multi_encoded = cloud_encoded + text_encoded
        output = self.pct_decoder(multi_encoded, eye_seed)
        return M.Model(inputs=[xyz, eye_seed, input_ids, attention_mask], outputs=output)

# Inference Function

### Example Usage of `run_inference`

To run inference with a single `.ply` file and class name:
```python
ply_file_path = 'path/to/cloud.ply'
class_name = 'example_class'
output = run_inference(AE, ply_file_path, class_name)
```
To run inference with a single NumPy array and class name:
```python
sample_point_cloud = np.random.rand(4096, 3)  # Example point cloud
class_name = 'example_class'
output = run_inference(AE, sample_point_cloud, class_name)
```
To run inference with multiple point clouds and class names:
```python
point_clouds = ['path/to/cloud1.ply', np.random.rand(4096, 3)]
class_names = ['class1', 'class2']
output = run_inference(AE, point_clouds, class_names)
```


In [8]:
def normalize_point_cloud(point_cloud):
    mean = np.mean(point_cloud, axis=0)
    point_cloud -= mean
    max_distance = np.max(np.sqrt(np.sum(point_cloud ** 2, axis=1)))
    point_cloud /= max_distance
    return point_cloud

def run_inference(model, point_cloud_input, class_name):
    """
    Runs inference using a provided model with a single or batch of point clouds and corresponding class names.
    The point clouds can be provided as file paths, NumPy arrays, or TensorFlow tensors.

    Args:
        model: The model to be used for inference.
        point_cloud_input (str, np.array, tf.Tensor, or list): A single PLY file path, 2D NumPy array,
                                                               TensorFlow tensor, or a list of these.
        class_name (str or list): A single class name or a list of class names, one for each point cloud.
    
    Returns:
        The output from the model.
    """
    # Ensure point_cloud_input and class_name are lists for batch processing
    if not isinstance(point_cloud_input, list):
        point_cloud_input = [point_cloud_input]
    if not isinstance(class_name, list):
        class_name = [class_name]

    if len(point_cloud_input) != len(class_name):
        raise ValueError("The number of point clouds must match the number of class names.")

    # Initialize lists to hold processed point clouds and tokenized text
    processed_point_clouds = []
    input_ids_list = []
    attention_mask_list = []

    # Initialize BERT tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    # Process each point cloud and class name
    for pc_input, cls_name in zip(point_cloud_input, class_name):
        # Check the type of the point cloud input and process accordingly
        if isinstance(pc_input, str):
            # Load the point cloud from the .ply file using trimesh
            mesh = trimesh.load(pc_input)
            point_cloud = np.array(mesh.vertices)
        elif isinstance(pc_input, np.ndarray):
            point_cloud = pc_input
        elif isinstance(pc_input, tf.Tensor):
            point_cloud = pc_input.numpy()
        else:
            raise TypeError("Each point cloud input should be a file path (str), a 2D NumPy array, or a TensorFlow tensor.")

        # Check that the point cloud has the correct shape
        if point_cloud.ndim != 2 or point_cloud.shape[1] != 3:
            raise ValueError("Each point cloud should be of shape (N, 3), where N is the number of points.")

        # Normalize the point cloud
        point_cloud = normalize_point_cloud(point_cloud)

        # Sample points using farthest point sampling
        partial_indices = fpsample.fps_sampling(point_cloud, 4096, start_idx=0)
        sampled_cloud = point_cloud[partial_indices]
        processed_point_clouds.append(sampled_cloud)

        # Tokenize the class name using BERT
        encoded_text = tokenizer.encode_plus(
            cls_name,
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='tf'
        )
        input_ids_list.append(encoded_text['input_ids'])
        attention_mask_list.append(encoded_text['attention_mask'])

    # Convert lists to tensors
    sample_point_clouds = tf.convert_to_tensor(processed_point_clouds, dtype=tf.float32)
    input_ids = tf.concat(input_ids_list, axis=0)
    attention_mask = tf.concat(attention_mask_list, axis=0)
    eye_seed = tf.zeros([len(point_cloud_input), 1, 1], dtype=tf.float32)

    # Call the model
    output = model([sample_point_clouds, eye_seed, input_ids, attention_mask])

    return output

## Visualization function

In [9]:

def visualize_cloud(point_cloud, title="Point Cloud Visualization"):
    """
    Visualizes a 3D point cloud using Plotly. The input can be a .ply file, a 2D/3D NumPy array,
    or a 2D/3D TensorFlow tensor.

    Args:
        point_cloud (str, np.array, or tf.Tensor): Path to a .ply file or a 2D/3D NumPy array or
                                                   TensorFlow tensor of shape (N, 3) or (1, N, 3),
                                                   where N is the number of points.
        title (str): Title of the plot.
    """
    # Check if the input is a string (assume it's a file path)
    if isinstance(point_cloud, str):
        # Load the point cloud from the .ply file using trimesh
        mesh = trimesh.load(point_cloud)
        point_cloud = np.array(mesh.vertices)
    elif isinstance(point_cloud, (np.ndarray, tf.Tensor)):
        # Convert TensorFlow tensor to NumPy array if needed
        if isinstance(point_cloud, tf.Tensor):
            point_cloud = point_cloud.numpy()
        
        # Handle the case where point_cloud has a batch dimension
        if point_cloud.ndim == 3:
            if point_cloud.shape[0] > 1:
                raise ValueError("Only one point cloud can be visualized at a time. Please provide a single point cloud.")
            elif point_cloud.shape[0] == 1:
                # Discard the batch dimension
                point_cloud = point_cloud[0]
        elif point_cloud.ndim != 2 or point_cloud.shape[1] != 3:
            raise ValueError("Point cloud should be of shape (N, 3) or (1, N, 3), where N is the number of points.")
    else:
        raise TypeError("point_cloud should be a file path (str), a 2D/3D NumPy array, or a TensorFlow tensor.")

    # Extract x, y, and z coordinates
    x, y, z = point_cloud[:, 0], point_cloud[:, 1], point_cloud[:, 2]

    # Create a 3D scatter plot using Plotly
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=x,
                y=y,
                z=z,
                mode='markers',
                marker=dict(
                    size=2,  # Size of the markers
                    color=z,  # Use the z-values for color
                    colorscale='Viridis',  # Color scale
                    opacity=0.8
                )
            )
        ]
    )

    # Set the layout of the plot
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X Axis',
            yaxis_title='Y Axis',
            zaxis_title='Z Axis'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

# We now can use the model

## Instantiating the model

In [10]:
AE = PCT_AE_Multimodal(bert_model=bert_model, PCT_encoder=PCT_encoder, pct_decoder=pct_decoder)
AE = AE.model

2026-07-27 04:43:54.942356: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22149 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:03:00.0, compute capability: 8.9
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassif

## Loading the Weights

In [11]:
AE.load_weights("MSN_weights3.h5")

## Running inference on a partial point cloud 

In [12]:

partial_path = "../../data/14161307/SkullFix/point_clouds/incomplete/skull_000_incomplete.ply"
output = run_inference(AE, partial_path, class_name= 'skull')


2026-07-27 04:44:01.220706: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904


## Visualizing the results

In [13]:

visualize_cloud(output, title = "Skull pred")


In [14]:
visualize_cloud(partial_path, title = "Skull partial input")

In [15]:
visualize_cloud("../../data/14161307/SkullFix/point_clouds/incomplete/skull_000_incomplete.ply")

# Quantitative Evaluation on the full SkullFix set

只看单个样本的可视化很难判断补全效果好不好，这里在 `data/14161307/SkullFix/point_clouds/` 下**全部** incomplete/complete 配对样本上跑一遍推理，用训练 notebook（`MSN_model_training_Demo.ipynb`）里同样定义的 **Chamfer Distance（`calc_cd`）** 和 **Density-aware Chamfer Distance（`calc_dcd`）** 来量化。

说明：
- 模型的解码器架构固定输出 6144 个点（`pct_decoder` 里 1024 → 3072 → 6144），所以 GT 也用 `fpsample` 采样到 6144 个点，才能跟模型输出的点数对齐、逐点算最近距离。
- GT 和 partial 输入分别各自独立做 `normalize_point_cloud`（跟 `run_inference` 现有逻辑一致），没有做额外的统一坐标对齐处理。
- 暂不涉及训练集/测试集划分是否有重叠的问题（按之前讨论，先不管这个）。

In [16]:
def eye_seed(X):
    return tf.zeros([X.shape[0],1,1])

def distance_matrix(array1, array2):
    batch_size, num_point, num_features = array1.shape
    expanded_array1 = tf.tile(tf.expand_dims(array1, 2), [1, 1, num_point, 1])
    expanded_array2 = tf.tile(tf.expand_dims(array2, 1), [1, num_point, 1, 1])
    distances = tf.norm(expanded_array1-expanded_array2, axis=-1)
    return distances

def min_distances_and_indices(array1, array2):
    distances = distance_matrix(array1, array2)
    min_dists_1_to_2, indices_1_to_2 = tf.reduce_min(distances, axis=-1), tf.argmin(distances, axis=-1)
    min_dists_2_to_1, indices_2_to_1 = tf.reduce_min(distances, axis=-2), tf.argmin(distances, axis=-2)
    return min_dists_1_to_2, min_dists_2_to_1, indices_1_to_2, indices_2_to_1

def calc_cd(output, gt, calc_f1=False, return_raw=False, normalize=False, separate=False):
    dist1, dist2, idx1, idx2 = min_distances_and_indices(gt, output)
    cd_p = (tf.sqrt(tf.reduce_mean(dist1, axis=1)) + tf.sqrt(tf.reduce_mean(dist2, axis=1))) / 2
    cd_t = (tf.reduce_mean(dist1, axis=1) + tf.reduce_mean(dist2, axis=1))
    if separate:
        res = [tf.concat([tf.reduce_mean(tf.sqrt(dist1), axis=1, keepdims=True),
                          tf.reduce_mean(tf.sqrt(dist2), axis=1, keepdims=True)], axis=0),
               tf.concat([tf.reduce_mean(dist1, axis=1, keepdims=True),
                          tf.reduce_mean(dist2, axis=1, keepdims=True)], axis=0)]
    else:
        res = [cd_p, cd_t]
    if calc_f1:
        f1, _, _ = fscore(dist1, dist2, 0.0001)
        res.append(f1)
    if return_raw:
        res.extend([dist1, dist2, idx1, idx2])
    return res

def calc_dcd(x, gt, alpha=1, n_lambda=1, return_raw=False, non_reg=False):
    x = tf.cast(x, tf.float32)
    gt = tf.cast(gt, tf.float32)
    batch_size = tf.shape(x)[0]
    n_x = tf.shape(x)[1]
    n_gt = tf.shape(gt)[1]
    if non_reg:
        frac_12 = tf.maximum(1.0, tf.cast(n_x, tf.float32) / tf.cast(n_gt, tf.float32))
        frac_21 = tf.maximum(1.0, tf.cast(n_gt, tf.float32) / tf.cast(n_x, tf.float32))
    else:
        frac_12 = tf.cast(n_x, tf.float32) / tf.cast(n_gt, tf.float32)
        frac_21 = tf.cast(n_gt, tf.float32) / tf.cast(n_x, tf.float32)
    cd_p, cd_t, dist1, dist2, idx1, idx2 = calc_cd(x, gt, return_raw=True)
    exp_dist1 = tf.exp(-dist1 * alpha)
    exp_dist2 = tf.exp(-dist2 * alpha)
    def compute_loss(b):
        idx1_b = tf.gather(idx1, b)
        idx2_b = tf.gather(idx2, b)
        count1 = tf.math.bincount(idx1_b, minlength=tf.cast(n_x, tf.int64))
        weight1 = tf.gather(count1, idx1_b)
        weight1 = tf.cast(weight1, tf.float32)
        weight1 = tf.pow(weight1, n_lambda)
        weight1 = tf.pow((weight1 + 1e-6), -1) * frac_21
        loss1 = tf.reduce_mean(-exp_dist1[b] * weight1 + 1.0)
        count2 = tf.math.bincount(idx2_b, minlength=tf.cast(n_gt, tf.int64))
        weight2 = tf.gather(count2, idx2_b)
        weight2 = tf.cast(weight2, tf.float32)
        weight2 = tf.pow(weight2, n_lambda)
        weight2 = tf.pow((weight2 + 1e-6), -1) * frac_12
        loss2 = tf.reduce_mean(-exp_dist2[b] * weight2 + 1.0)
        return loss1, loss2
    loss1, loss2 = tf.map_fn(compute_loss, tf.range(batch_size), dtype=(tf.float32, tf.float32))
    loss = tf.reduce_mean(loss1 + loss2)
    res = [loss, cd_p, cd_t]
    if return_raw:
        res.extend([dist1, dist2, idx1, idx2])
    return loss

In [17]:
import os
import glob
import pandas as pd
from tqdm import tqdm

skullfix_root = "../../data/14161307/SkullFix/point_clouds"
N_SAMPLES = 50  # 数据量减到 50 个，跑起来更快
incomplete_files = sorted(glob.glob(os.path.join(skullfix_root, "incomplete", "*.ply")))[:N_SAMPLES]
print(f"共找到 {len(sorted(glob.glob(os.path.join(skullfix_root, 'incomplete', '*.ply'))))} 个样本，本次评估取前 {len(incomplete_files)} 个")

eval_results = []
for incomplete_path in tqdm(incomplete_files, desc="Evaluating on SkullFix"):
    sample_id = os.path.basename(incomplete_path).replace("_incomplete.ply", "")
    complete_path = os.path.join(skullfix_root, "complete", f"{sample_id}_complete.ply")
    if not os.path.exists(complete_path):
        print(f"跳过 {sample_id}：找不到对应的 complete 文件")
        continue

    # 这台机器的 24GB 显存被 BERT+PCT 模型加载完就快占满了，试了 cuda_malloc_async
    # 和 set_memory_growth 都还是会在推理某一步 OOM。评估这一步只是离线跑一次算指标，
    # 不追求实时，干脆整个推理 + 指标计算都放 CPU 上跑，彻底不碰 GPU，保证能跑完。
    # CPU 有 46GB 内存，跑几千个点规模的计算完全没问题，就是比 GPU 慢一些。
    with tf.device('/CPU:0'):
        # 模型预测：复用现成的 run_inference（内部已经做了归一化 + FPS 采样 + 推理），固定输出 6144 个点
        pred = run_inference(AE, incomplete_path, class_name='skull')

        # GT：同样先归一化，再用 FPS 采样到 6144 个点，才能跟模型输出的点数对齐算距离
        gt_mesh = trimesh.load(complete_path)
        gt_points = normalize_point_cloud(np.array(gt_mesh.vertices))
        gt_idx = fpsample.fps_sampling(gt_points, 6144, start_idx=0)
        gt_points = gt_points[gt_idx]
        gt_tensor = tf.convert_to_tensor(gt_points[None, ...], dtype=tf.float32)

        cd_p, cd_t = calc_cd(pred, gt_tensor)
        dcd = calc_dcd(pred, gt_tensor)

    eval_results.append({
        "id": sample_id,
        "CD_p": float(cd_p.numpy()[0]),
        "CD_t": float(cd_t.numpy()[0]),
        "DCD": float(dcd.numpy()),
    })

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv("skullfix_eval_results.csv", index=False)
print(f"评估完成，成功 {len(eval_df)} / {len(incomplete_files)} 个样本，结果存到 skullfix_eval_results.csv")
eval_df

共找到 100 个样本，本次评估取前 50 个


Evaluating on SkullFix: 100%|██████████| 50/50 [05:10<00:00,  6.22s/it]

评估完成，成功 50 / 50 个样本，结果存到 skullfix_eval_results.csv


,id,CD_p,CD_t,DCD
0,skull_000,0.199880,0.080934,1.437818
1,skull_001,0.226680,0.102793,1.424618
2,skull_002,0.222623,0.099449,1.431655
3,skull_003,0.213645,0.091354,1.418796
4,skull_004,0.211065,0.089257,1.394665
5,skull_005,0.224967,0.101266,1.417585
6,skull_006,0.217164,0.094322,1.414608
7,skull_007,0.217796,0.095040,1.429316
8,skull_008,0.202651,0.082637,1.426167
9,skull_009,0.194274,0.076410,1.433411


In [18]:
# 统计汇总
print(f"评估样本数: {len(eval_df)}")
display(eval_df[["CD_p", "CD_t", "DCD"]].describe())

# 每个指标的分布（箱线图 + 每个点），一眼看出整体水平和离群样本
fig = go.Figure()
for metric in ["CD_p", "CD_t", "DCD"]:
    fig.add_trace(go.Box(y=eval_df[metric], name=metric, boxpoints="all", jitter=0.3))
fig.update_layout(title="SkullFix 全量评估指标分布", yaxis_title="指标值")
fig.show()

print("CD_p 最差的 5 个样本（补全效果最差，值得挑出来看看点云长什么样）：")
display(eval_df.sort_values("CD_p", ascending=False).head(5))
print("CD_p 最好的 5 个样本：")
display(eval_df.sort_values("CD_p", ascending=True).head(5))

评估样本数: 50


,CD_p,CD_t,DCD
count,50.000000,50.000000,50.000000
mean,0.220077,0.097433,1.430469
std,0.011294,0.009823,0.020353
min,0.194274,0.076410,1.391311
25%,0.211099,0.089414,1.418808
50%,0.221003,0.098324,1.430107
75%,0.227773,0.103893,1.439609
max,0.242468,0.118491,1.501321


CD_p 最差的 5 个样本（补全效果最差，值得挑出来看看点云长什么样）：


,id,CD_p,CD_t,DCD
40,skull_040,0.242468,0.118491,1.404662
26,skull_026,0.242395,0.117803,1.430287
48,skull_048,0.240673,0.115861,1.466844
16,skull_016,0.238049,0.113898,1.416047
20,skull_020,0.232721,0.108608,1.422681


CD_p 最好的 5 个样本：


,id,CD_p,CD_t,DCD
9,skull_009,0.194274,0.076410,1.433411
0,skull_000,0.199880,0.080934,1.437818
49,skull_049,0.202372,0.082875,1.429928
8,skull_008,0.202651,0.082637,1.426167
13,skull_013,0.205202,0.085191,1.435798
